In [ ]:
from google.colab import files
files.upload()  # Upload the kaggle.json file you just downloaded


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ousmanebagayoko","key":"7c000cb1ef943a96cadd7e5092f8864d"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d abdussamad123/user-daily-nutritional-intake

Dataset URL: https://www.kaggle.com/datasets/abdussamad123/user-daily-nutritional-intake
License(s): CC-BY-NC-SA-4.0
  0% 0.00/38.5k [00:00<?, ?B/s]
100% 38.5k/38.5k [00:00<00:00, 132MB/s]


In [ ]:
!unzip user-daily-nutritional-intake.zip -d data/

Archive:  user-daily-nutritional-intake.zip
  inflating: data/user_nutritional_data.csv  


In [ ]:
import pandas as pd

df = pd.read_csv("data/user_nutritional_data.csv")
df.head()  # shows the first few rows

,Gender,Age,Daily meals frequency,Physical exercise,Height,Weight,BMR,Carbs,Proteins,Fats,Calories
0,0,29,3,0,165,101.0,1901.25,285.188,114.075,76.050,2281.502
1,1,25,3,4,165,53.0,1275.25,302.872,121.149,80.766,2422.978
2,0,23,2,0,170,70.0,1652.50,247.875,99.150,66.100,1983.000
3,0,22,3,0,168,112.0,2065.00,309.750,123.900,82.600,2478.000
4,0,19,3,2,175,67.0,1673.75,324.289,129.716,86.477,2594.313


In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/user_nutritional_data.csv")

# Show basic information
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("Column Names and Types:")
print(df.info())
print("\n" + "="*50)
print("First few rows:")
print(df.head(10))
print("\n" + "="*50)
print("Statistical Summary:")
print(df.describe())
print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())

Dataset Shape: (2182, 11)

Column Names and Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2182 entries, 0 to 2181
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 2182 non-null   int64  
 1   Age                    2182 non-null   int64  
 2   Daily meals frequency  2182 non-null   int64  
 3   Physical exercise      2182 non-null   int64  
 4   Height                 2182 non-null   int64  
 5   Weight                 2182 non-null   float64
 6   BMR                    2182 non-null   float64
 7   Carbs                  2182 non-null   float64
 8   Proteins               2182 non-null   float64
 9   Fats                   2182 non-null   float64
 10  Calories               2182 non-null   float64
dtypes: float64(6), int64(5)
memory usage: 187.6 KB
None

First few rows:
   Gender  Age  Daily meals frequency  Physical exercise  Height  Weight  \
0       0   29  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv("data/user_nutritional_data.csv")

print("="*60)
print("CALORIE MAINTENANCE PREDICTOR - ML PIPELINE")
print("="*60)

# ===========================
# 1. DATA EXPLORATION
# ===========================
print("\n1. DATASET OVERVIEW")
print("-" * 60)
print(f"Total samples: {len(df)}")
print(f"Features: {df.shape[1]}")
print(f"\nTarget variable: Calories (mean={df['Calories'].mean():.2f}, std={df['Calories'].std():.2f})")

# ===========================
# 2. FEATURE SELECTION
# ===========================
print("\n2. FEATURE SELECTION")
print("-" * 60)

# Features to use for prediction (excluding BMR, macros, and target)
# We exclude BMR because it's derived from the same info we're predicting
# We exclude macros because they're outputs, not inputs
feature_columns = ['Gender', 'Age', 'Height', 'Weight', 'Physical exercise', 'Daily meals frequency']
target_column = 'Calories'

X = df[feature_columns]
y = df[target_column]

print(f"Selected features: {feature_columns}")
print(f"Target: {target_column}")

# ===========================
# 3. FEATURE CORRELATIONS
# ===========================
print("\n3. FEATURE CORRELATIONS WITH CALORIES")
print("-" * 60)
correlations = df[feature_columns + [target_column]].corr()[target_column].sort_values(ascending=False)
print(correlations)

# ===========================
# 4. TRAIN-TEST SPLIT
# ===========================
print("\n4. SPLITTING DATA")
print("-" * 60)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# ===========================
# 5. FEATURE SCALING
# ===========================
print("\n5. FEATURE SCALING")
print("-" * 60)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features normalized using StandardScaler")

# ===========================
# 6. MODEL TRAINING & EVALUATION
# ===========================
print("\n6. TRAINING MULTIPLE MODELS")
print("=" * 60)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
}

results = {}

for name, model in models.items():
    print(f"\n{name}")
    print("-" * 60)

    # Train the model
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results[name] = {
        'model': model,
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'predictions': y_pred
    }

    print(f"Mean Absolute Error (MAE): {mae:.2f} calories")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} calories")
    print(f"R² Score: {r2:.4f}")
    print(f"Average prediction error: ±{mae:.2f} calories/day")

# ===========================
# 7. BEST MODEL SELECTION
# ===========================
print("\n" + "="*60)
print("7. BEST MODEL SELECTION")
print("="*60)
best_model_name = min(results.keys(), key=lambda x: results[x]['mae'])
best_model = results[best_model_name]['model']
print(f"Best Model: {best_model_name}")
print(f"MAE: {results[best_model_name]['mae']:.2f} calories")
print(f"R² Score: {results[best_model_name]['r2']:.4f}")

# ===========================
# 8. FEATURE IMPORTANCE (for tree-based models)
# ===========================
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    print("\n8. FEATURE IMPORTANCE")
    print("-" * 60)
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    print(feature_importance.to_string(index=False))

# ===========================
# 9. PREDICTION FUNCTION
# ===========================
print("\n" + "="*60)
print("9. PREDICTION FUNCTION READY")
print("="*60)

def predict_calories(gender, age, height, weight, physical_exercise, daily_meals_freq):
    """
    Predict daily calorie maintenance for a person.

    Parameters:
    - gender: 0 for Female, 1 for Male
    - age: Age in years
    - height: Height in cm
    - weight: Weight in kg
    - physical_exercise: Activity level (0-4, where 0=sedentary, 4=very active)
    - daily_meals_freq: Number of meals per day (2-4)

    Returns:
    - Predicted daily calorie maintenance
    """
    input_data = pd.DataFrame({
        'Gender': [gender],
        'Age': [age],
        'Height': [height],
        'Weight': [weight],
        'Physical exercise': [physical_exercise],
        'Daily meals frequency': [daily_meals_freq]
    })

    if best_model_name == 'Linear Regression':
        input_scaled = scaler.transform(input_data)
        prediction = best_model.predict(input_scaled)[0]
    else:
        prediction = best_model.predict(input_data)[0]

    return round(prediction, 2)

# ===========================
# 10. EXAMPLE PREDICTIONS
# ===========================
print("\n10. EXAMPLE PREDICTIONS")
print("="*60)

examples = [
    (0, 25, 165, 60, 2, 3, "25yo Female, 165cm, 60kg, Moderate activity"),
    (1, 30, 180, 80, 3, 3, "30yo Male, 180cm, 80kg, Active"),
    (0, 22, 160, 55, 1, 2, "22yo Female, 160cm, 55kg, Light activity"),
    (1, 28, 175, 75, 4, 3, "28yo Male, 175cm, 75kg, Very active"),
]

for gender, age, height, weight, exercise, meals, description in examples:
    calories = predict_calories(gender, age, height, weight, exercise, meals)
    print(f"{description}")
    print(f"  → Predicted maintenance: {calories} calories/day\n")

print("="*60)
print("MODEL TRAINING COMPLETE!")
print("="*60)
print("\nYou can now use predict_calories() function to predict for new users.")
print("\nExample usage:")
print("  calories = predict_calories(gender=1, age=25, height=175, weight=70,")
print("                              physical_exercise=2, daily_meals_freq=3)")

CALORIE MAINTENANCE PREDICTOR - ML PIPELINE

1. DATASET OVERVIEW
------------------------------------------------------------
Total samples: 2182
Features: 11

Target variable: Calories (mean=2002.87, std=448.26)

2. FEATURE SELECTION
------------------------------------------------------------
Selected features: ['Gender', 'Age', 'Height', 'Weight', 'Physical exercise', 'Daily meals frequency']
Target: Calories

3. FEATURE CORRELATIONS WITH CALORIES
------------------------------------------------------------
Calories                 1.000000
Physical exercise        0.639275
Weight                   0.580019
Height                   0.552983
Daily meals frequency    0.091730
Age                     -0.220797
Gender                  -0.561667
Name: Calories, dtype: float64

4. SPLITTING DATA
------------------------------------------------------------
Training samples: 1745
Testing samples: 437

5. FEATURE SCALING
------------------------------------------------------------
Features n